In [1]:
import os
import platform
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.formula.translate import Translator
from openpyxl.utils import get_column_letter

In [2]:
INPUT_DIR = Path('头部公募基金')
OUTPUT_DIR = Path('头部公募基金整合')
COPY_COLUMNS_PATH = Path('公募基金复制列.xlsx')
OUTPUT_PATH = OUTPUT_DIR / '头部公募基金数据.xlsx'
WIND_SOURCE_MARKER = '数据来源:wind'

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f'未找到头部公募基金目录：{INPUT_DIR.resolve()}')
if not COPY_COLUMNS_PATH.is_file():
    raise FileNotFoundError(f'未找到公募基金复制列模板：{COPY_COLUMNS_PATH.resolve()}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
import time

def normalize_source_text(value):
    return str(value).strip().replace('：', ':').replace(' ', '').lower()


def read_headfund_report(file_path):
    print(f'正在提取：{file_path}',end=' ')
    s = time.time()
    with pd.ExcelFile(file_path) as workbook:
        if 'sheet1' not in workbook.sheet_names:
            raise KeyError(f'{file_path.name} 缺少 sheet1')
        dataframe = pd.read_excel(workbook, sheet_name='sheet1')

    if len(dataframe) < 2:
        raise ValueError(f'{file_path.name} 不足两行，无法核验 Wind 来源', end=' ')

    footer_values = dataframe.tail(2).fillna('').astype(str).to_numpy().ravel()
    has_wind_source = any(
        WIND_SOURCE_MARKER in normalize_source_text(value)
        for value in footer_values
    )
    if not has_wind_source:
        raise ValueError(f'{file_path.name} 末两行未检测到“数据来源：Wind”', end=' ')

    # 所有年报/中报均排除最后两行：空行与 Wind 来源说明。
    dataframe = dataframe.iloc[:-2].copy()
    dataframe['文件名'] = file_path.stem
    e = time.time()
    spt = round(e-s,3)
    ape = '' if spt > 1 else '*'
    print(f'用时{spt}秒 {ape}')
    return dataframe


def load_copy_column_specs(template_path):
    workbook = load_workbook(
        template_path,
        read_only=True,
        data_only=False,
        keep_links=True,
    )
    worksheet = workbook[workbook.sheetnames[0]]
    specs = []
    for column_index in range(1, worksheet.max_column + 1):
        header = worksheet.cell(1, column_index).value
        if header is not None:
            specs.append((column_index, header, worksheet.cell(2, column_index).value))
    workbook.close()

    if not specs:
        raise ValueError(f'{template_path} 没有可复制列')
    if '计数' not in {header for _, header, _ in specs}:
        raise KeyError(f'{template_path} 缺少“计数”列')
    return specs


def fill_copy_columns(dataframe, specs):
    result = dataframe.reset_index(drop=True).copy()
    data_row_count = len(result)

    for column_index, header, template_value in specs:
        insert_at = column_index - 1
        if insert_at > len(result.columns):
            raise ValueError(
                f'复制列“{header}”应位于第 {column_index} 列，'
                f'但当前数据只有 {len(result.columns)} 列'
            )
        if header in result.columns:
            raise ValueError(f'汇总数据已存在重复列：{header}')

        column_letter = get_column_letter(column_index)
        if header == '计数':
            values = [1] * data_row_count
        else:
            if not (isinstance(template_value, str) and template_value.startswith('=')):
                raise ValueError(f'复制列模板的“{header}”第2行不是公式')
            origin = f'{column_letter}2'
            translator = Translator(template_value, origin=origin)
            values = [
                translator.translate_formula(f'{column_letter}{excel_row}')
                for excel_row in range(2, data_row_count + 2)
            ]

        result.insert(insert_at, header, values)
        print(f'复制列完成：{column_letter} {header}')

    return result


report_paths = sorted(
    path for path in INPUT_DIR.rglob('*')
    if path.is_file()
    and path.suffix.lower() in {'.xls', '.xlsx'}
    and not path.name.startswith('~$')
)
if not report_paths:
    raise FileNotFoundError(f'{INPUT_DIR.resolve()} 中没有 Excel 年报数据')

df_concat = []
skipped_files = []
for report_path in report_paths:
    try:
        df_concat.append(read_headfund_report(report_path))
    except Exception as error:
        skipped_files.append((report_path, error))
        print(f'跳过文件：{report_path}；原因：{error}')

if not df_concat:
    raise RuntimeError('没有通过 Wind 来源核验的公募基金数据')

df0 = pd.concat(df_concat, ignore_index=True)
copy_column_specs = load_copy_column_specs(COPY_COLUMNS_PATH)
df0 = fill_copy_columns(df0, copy_column_specs)
print(
    f'汇总完成：{len(df_concat)}/{len(report_paths)} 个文件，'
    f'{len(df0)} 行；跳过 {len(skipped_files)} 个文件'
)

正在提取：头部公募基金\华夏25年中.xlsx 用时1.948秒 
正在提取：头部公募基金\华夏25年末.xlsx 用时1.981秒 
正在提取：头部公募基金\华夏26年中.xlsx 用时0.491秒 *
正在提取：头部公募基金\南方25年中.xlsx 用时2.627秒 
正在提取：头部公募基金\南方25年末.xlsx 用时2.08秒 
正在提取：头部公募基金\南方26年中.xlsx 用时0.385秒 *
正在提取：头部公募基金\博时25年中.xlsx 用时1.658秒 
正在提取：头部公募基金\博时25年末.xlsx 用时1.644秒 
正在提取：头部公募基金\博时26年中.xlsx 用时0.353秒 *
正在提取：头部公募基金\嘉实25年中.xlsx 用时1.562秒 
正在提取：头部公募基金\嘉实25年末.xlsx 用时1.464秒 
正在提取：头部公募基金\嘉实26年中.xlsx 用时0.285秒 *
正在提取：头部公募基金\天弘25年中.xlsx 用时2.815秒 
正在提取：头部公募基金\天弘25年末.xlsx 用时2.43秒 
正在提取：头部公募基金\天弘26年中.xlsx 用时0.437秒 *
正在提取：头部公募基金\富国25年中.xlsx 用时1.792秒 
正在提取：头部公募基金\富国25年末.xlsx 用时2.156秒 
正在提取：头部公募基金\富国26年中.xlsx 用时0.54秒 *
正在提取：头部公募基金\广发25年中.xlsx 用时1.854秒 
正在提取：头部公募基金\广发25年末.xlsx 用时1.862秒 
正在提取：头部公募基金\广发26年中.xlsx 用时0.426秒 *
正在提取：头部公募基金\易方达25年中.xlsx 用时1.81秒 
正在提取：头部公募基金\易方达25年末.xlsx 用时1.795秒 
正在提取：头部公募基金\易方达26年中.xlsx 用时0.334秒 *
正在提取：头部公募基金\汇添富25年中.xlsx 用时1.464秒 
正在提取：头部公募基金\汇添富25年末.xlsx 用时2.184秒 
正在提取：头部公募基金\汇添富26年中.xlsx 用时0.283秒 *
正在提取：头部公募基金\鹏华25年中.xlsx 用时1.42秒 
正在提取：头部公募基金\鹏华25年末.xlsx 用时1.896秒 
正在提取

In [ ]:
df0.to_excel(OUTPUT_PATH, index=False, engine='openpyxl')
print(f'已保存：{OUTPUT_PATH.resolve()}')


def open_file(file_path):
    if platform.system() == "Windows":
        os.startfile(file_path)
    elif platform.system() == "Darwin":
        os.system(f'open "{file_path}"')
    elif platform.system() == "Linux":
        os.system(f'xdg-open "{file_path}"')
    else:
        print(f"无法在 {platform.system()} 系统上打开文件。请手动查找并打开 {file_path}。")

open_file(OUTPUT_PATH)